In [1]:
import json, random, re, time, math, statistics, os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import precision_score, recall_score, f1_score
from datetime import datetime
import ollama
from rank_bm25 import BM25Okapi

# Try to import SelfCheckGPT (optional - if not installed, use fallback)
try:
    from selfcheckgpt.modeling_selfcheck import SelfCheckNgram
    SELFCHECK_AVAILABLE = True
except ImportError:
    SELFCHECK_AVAILABLE = False
    print("Warning: selfcheckgpt not installed. Run: pip install selfcheckgpt")

# ============================================================
# CONFIGURATION (Small sample for quick testing)
# ============================================================
FEVER_PATH = "datasets/fever.jsonl"
HALU_PATH = "datasets/halueval.json"
TRUTH_PATH = "datasets/TruthfulQA.csv"

LLM_MODEL = "gemma3:1b"
SAMPLE_PER_DATASET = 500  # Small for quick testing
SEED = 42

UNCERTAINTY_THRESHOLD = 0.3
OVERLAP_THRESHOLD = 0.5
SCORE_THRESHOLD = 6.2

random.seed(SEED)
np.random.seed(SEED)
os.makedirs("results", exist_ok=True)

# ============================================================
# BM25 RETRIEVER (CPU-friendly)
# ============================================================
def tokenize(text):
    return [w.lower() for w in re.split(r"\s+", str(text)) if w]

def build_bm25_index(corpus):
    """Build BM25 index from corpus"""
    tokenized_corpus = [tokenize(doc) for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    return bm25, corpus  # Return both the index and the original corpus

def retrieve_bm25(claim, bm25_index, corpus, top_k=3):
    """Retrieve top-k relevant documents using BM25"""
    tokenized_claim = tokenize(claim)
    scores = bm25_index.get_scores(tokenized_claim)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [corpus[i] for i in top_indices if scores[i] > 0]

# ============================================================
# HELPERS
# ============================================================
def build_unigram(corpus):
    tokens = [t for doc in corpus for t in tokenize(doc)]
    counts = Counter(tokens)
    total = sum(counts.values()) or 1
    return counts, total

def unigram_neglogprob(text, counts, total):
    toks = tokenize(text)
    if not toks:
        return 0.0
    return sum(-math.log(counts.get(t, 1) / total) for t in toks) / len(toks)

def evidence_overlap(claim, docs):
    cs = set(tokenize(claim))
    if not cs or not docs:
        return 0.0
    return max((len(cs & set(tokenize(d))) / len(cs) for d in docs), default=0.0)

def get_predictive_entropy(text, model=LLM_MODEL):
    try:
        response = ollama.generate(
            model=model,
            prompt=text,
            options={"temperature": 0.0, "num_predict": 5, "logprobs": True, "top_logprobs": 5}
        )
        if hasattr(response, 'logprobs') and response.logprobs:
            all_entropy = []
            for token_logprobs in response.logprobs:
                if token_logprobs and 'top_logprobs' in token_logprobs:
                    probs = [np.exp(lp) for lp in token_logprobs['top_logprobs'].values()]
                    if probs:
                        probs = np.array(probs) / np.sum(probs)
                        entropy = -np.sum(probs * np.log(probs + 1e-10))
                        all_entropy.append(entropy)
            if all_entropy:
                return min(1.0, np.mean(all_entropy) / 5.0)
        return 0.5
    except:
        return 0.5

def call_llm_verify(claim, docs):
    evidence = " ".join(docs[:3])[:500] if docs else "No evidence"
    prompt = f"Evidence: {evidence}\nClaim: {claim}\nAnswer ONLY: SUPPORT or REFUTE"
    try:
        response = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}], options={"temperature": 0.0})
        return 0 if "SUPPORT" in response['message']['content'].upper() else 1
    except:
        return 1

# ============================================================
# VICTOR FRAMEWORK (with BM25 retrieval)
# ============================================================
def victor_full(claim, docs, rc, rt, use_uncertainty, bm25_index, bm25_corpus):
    parts = re.split(r'[.!?]+', claim)
    parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
    preds = []
    for p in parts:
        if not p:
            continue
        if use_uncertainty:
            uncertainty = get_predictive_entropy(p)
            if uncertainty > UNCERTAINTY_THRESHOLD:
                if bm25_index:
                    retrieved_docs = retrieve_bm25(p, bm25_index, bm25_corpus, top_k=3)
                    pred = call_llm_verify(p, retrieved_docs)
                else:
                    pred = call_llm_verify(p, docs)
            else:
                pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        else:
            overlap = evidence_overlap(p, docs)
            score = unigram_neglogprob(p, rc, rt)
            pred = 0 if overlap >= OVERLAP_THRESHOLD else (1 if score > SCORE_THRESHOLD else 1)
        preds.append(pred)
    return 1 if any(preds) else 0

def victor_no_unc(claim, docs, rc, rt):
    """VICTOR without uncertainty (uses BM25 if available)"""
    # Pass None for bm25_index and bm25_corpus to skip uncertainty
    return victor_full(claim, docs, rc, rt, use_uncertainty=False, bm25_index=None, bm25_corpus=None)

# ============================================================
# BASELINES
# ============================================================
def baseline_factscore(claim, docs, rc, rt):
    return 0 if evidence_overlap(claim, docs) >= 0.4 else 1

def baseline_rag(claim, docs, rc, rt):
    return 0 if evidence_overlap(claim, docs) >= 0.35 else 1

def baseline_selfcheckgpt(claim, docs, rc, rt):
    """Use SelfCheckGPT if available, fallback to unigram"""
    if SELFCHECK_AVAILABLE:
        try:
            selfcheck = SelfCheckNgram(n=1)
            sentences = [claim]
            sampled_passages = [docs[0]] if docs else [claim]
            scores = selfcheck.predict(sentences=sentences, sampled_passages=sampled_passages)
            # Higher score = more hallucinated
            hallucination_score = scores.get('sent_level', {}).get('avg_neg_logprob', [0])[0]
            return 1 if hallucination_score > 3.0 else 0
        except:
            return 1 if unigram_neglogprob(claim, rc, rt) > 5.5 else 0
    else:
        return 1 if unigram_neglogprob(claim, rc, rt) > 5.5 else 0

def baseline_verillm(claim, docs, rc, rt):
    overlap = evidence_overlap(claim, docs)
    score = unigram_neglogprob(claim, rc, rt)
    return 0 if overlap >= 0.3 else (1 if score > 6.0 else 1)

def baseline_relic(claim, docs, rc, rt):
    sc = 1 if unigram_neglogprob(claim, rc, rt) > 5.5 else 0
    vl = baseline_verillm(claim, docs, rc, rt)
    return 1 if (sc == 1 and vl == 1) else 0

# ============================================================
# DATA LOADERS
# ============================================================
def load_fever(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            claim, label = obj.get('claim', ''), obj.get('label', '').upper()
            if label == 'SUPPORTS':
                data.append((claim, 0, []))
            elif label == 'REFUTES':
                data.append((claim, 1, []))
    random.shuffle(data)
    return data[:SAMPLE_PER_DATASET]

def load_halueval(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if obj.get('right_response'):
                    data.append((obj['right_response'], 0, []))
                if obj.get('hallucinated_response'):
                    data.append((obj['hallucinated_response'], 1, []))
            except:
                continue
    random.shuffle(data)
    return data[:SAMPLE_PER_DATASET]

def load_truthfulqa(path):
    data = []
    df = pd.read_csv(path)
    for _, row in df.iterrows():
        if str(row.get('Best Answer', '')).strip():
            data.append((str(row['Best Answer']), 0, []))
        if str(row.get('Incorrect Answers', '')).strip():
            for inc in str(row['Incorrect Answers']).split(';'):
                data.append((inc.strip(), 1, []))
    random.shuffle(data)
    return data[:SAMPLE_PER_DATASET]

# ============================================================
# METRICS
# ============================================================
def safe_metrics(y_true, y_pred):
    if len(set(y_true)) < 2 or len(set(y_pred)) < 2:
        correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
        if correct == len(y_true):
            return 1.0, 1.0, 1.0
        else:
            return 0.0, 0.0, 0.0
    return (
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0)
    )

def bootstrap_ci(y_true, y_pred, n_bootstrap=500, ci=95):
    """Compute confidence interval for F1 score"""
    f1_scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        _, _, f1 = safe_metrics([y_true[i] for i in idx], [y_pred[i] for i in idx])
        f1_scores.append(f1)
    lower = np.percentile(f1_scores, (100 - ci) / 2)
    upper = np.percentile(f1_scores, 100 - (100 - ci) / 2)
    return lower, upper

def evaluate_framework(name, fn, dataset, ref_docs, rc, rt, use_uncertainty=True, bm25_index=None, bm25_corpus=None):
    y_true, y_pred = [], []
    for claim, label, docs in dataset:
        if "No Uncertainty" in name:
            pred = fn(claim, ref_docs, rc, rt)
        elif name.startswith("VICTOR"):
            pred = fn(claim, ref_docs, rc, rt, use_uncertainty, bm25_index, bm25_corpus)
        else:
            pred = fn(claim, ref_docs, rc, rt)
        y_true.append(label)
        y_pred.append(pred)
    
    p, r, f = safe_metrics(y_true, y_pred)
    ci_lower, ci_upper = bootstrap_ci(y_true, y_pred)
    hallucination_rate = 1 - p if p > 0 else 1
    
    return {
        "Framework": name, 
        "Precision": round(p, 3), 
        "Recall": round(r, 3), 
        "F1 Score": round(f, 3), 
        "F1 CI": f"[{ci_lower:.3f}-{ci_upper:.3f}]",
        "Hallucination Rate": round(hallucination_rate, 3)
    }

# ============================================================
# MAIN
# ============================================================
def main():
    print("\n" + "=" * 70)
    print("VICTOR - Verification with Intelligent Confidence Thresholding and Optimal Retrieval")
    print(f"Model: {LLM_MODEL} | Samples: {SAMPLE_PER_DATASET} | τ: {UNCERTAINTY_THRESHOLD}")
    print(f"BM25: Enabled (Training-only) | SelfCheckGPT: {'Available' if SELFCHECK_AVAILABLE else 'Fallback mode'}")
    print("=" * 70)
    
    print("\nParameter\tValue\tNote")
    print("-" * 55)
    print(f"SAMPLE_PER_DATASET\t{SAMPLE_PER_DATASET}\tTotal records loaded per source")
    print(f"TAU (Uncertainty Gate)\t{UNCERTAINTY_THRESHOLD}\tThreshold for LLM trigger")
    print(f"SEED\t{SEED}\tFor reproducibility")
    
    # Load datasets
    fever = load_fever(FEVER_PATH)
    halu = load_halueval(HALU_PATH)
    truth = load_truthfulqa(TRUTH_PATH)
    datasets = {"FEVER": fever, "HaluEval": halu, "TruthfulQA": truth}
    
    total_test_claims = sum(len(d) // 2 for d in datasets.values())
    print(f"Total Test Claims\t{total_test_claims}\t50% split used for testing")
    
    all_results = []
    frameworks = {
        "FActScore": baseline_factscore,
        "RAG": baseline_rag,
        "SelfCheckGPT": baseline_selfcheckgpt,
        "VeriLLM": baseline_verillm,
        "ReLiC": baseline_relic,
        "VICTOR (Ours)": victor_full
    }
    
    for ds_name, ds_data in datasets.items():
        print(f"\n  Processing {ds_name}...")
        split = len(ds_data) // 2
        train, test = ds_data[:split], ds_data[split:]
        
        # Build reference from TRAINING claims only (NO DATA LEAKAGE)
        train_ref = [c for c, l, _ in train if l == 0]
        ds_rc, ds_rt = build_unigram(train_ref) if train_ref else ({}, 1)
        
        # Build BM25 index from TRAINING claims only
        bm25_index, bm25_corpus = build_bm25_index(train_ref) if train_ref else (None, None)
        print(f"    Training claims: {len(train)} | Test claims: {len(test)} | BM25 docs: {len(train_ref)}")
        
        for fw_name, fw_fn in frameworks.items():
            use_unc = (fw_name == "VICTOR (Ours)")
            res = evaluate_framework(fw_name, fw_fn, test, train_ref, ds_rc, ds_rt, use_unc, bm25_index, bm25_corpus)
            res["Dataset"] = ds_name
            all_results.append(res)
    
    # Rest of the function remains the same (printing, saving, etc.)
    # ... (keep your existing printing and saving code)
    
    df = pd.DataFrame(all_results)
    # DEBUG: Check what's in the DataFrame
    print("\n" + "=" * 70)
    print("DEBUG: DataFrame content (first 10 rows)")
    print("=" * 70)
    print(df[["Dataset", "Framework", "F1 Score"]].head(10))
    print(f"\nF1 Score column type: {df['F1 Score'].dtype}")
    
    # Ensure F1 Score is numeric
    df['F1 Score'] = pd.to_numeric(df['F1 Score'], errors='coerce')
    
    # ============================================================
    # PRINT RESULTS
    # ============================================================
    print("\n\n" + "=" * 70)
    print("PER DATASET RESULTS")
    print("=" * 70)
    print("Dataset\tFramework\tPrecision\tRecall\tF1 Score\tHall Rate\t95% CI")
    for _, row in df.iterrows():
        print(f"{row['Dataset']}\t{row['Framework']}\t{row['Precision']}\t{row['Recall']}\t{row['F1 Score']}\t{row['Hallucination Rate']}\t{row['F1 CI']}")    
    # ============================================================
    # PIVOT TABLE (F1 Score by Dataset) - Fixed
    # ============================================================
    # Create pivot with numeric F1 scores
    pivot = df.pivot(index="Dataset", columns="Framework", values="F1 Score").reset_index()
    
    # Fill NaN with 0
    pivot = pivot.fillna(0)
    
    # Define desired column order
    desired_order = ["Dataset", "FActScore", "RAG", "SelfCheckGPT", "VeriLLM", "ReLiC", "VICTOR (Ours)"]
    
    # Reorder columns
    pivot = pivot[desired_order]
    
    print("\n\n" + "=" * 70)
    print("PIVOT TABLE (F1 Score by Dataset)")
    print("=" * 70)
    print("Dataset\tFActScore\tRAG\tSelfCheckGPT\tVeriLLM\tReLiC\tVICTOR")
    for _, row in pivot.iterrows():
        print(f"{row['Dataset']}\t{row['FActScore']:.3f}\t{row['RAG']:.3f}\t{row['SelfCheckGPT']:.3f}\t{row['VeriLLM']:.3f}\t{row['ReLiC']:.3f}\t{row['VICTOR (Ours)']:.3f}")
    # ============================================================
    # OVERALL AVERAGES
    # ============================================================
        # ============================================================
    # OVERALL AVERAGES (Fixed Order)
    # ============================================================
    overall = df.groupby("Framework").agg({"Precision": "mean", "Recall": "mean", "F1 Score": "mean", "Hallucination Rate": "mean"}).round(3).reset_index()
    
    # Define desired order
    desired_framework_order = ["FActScore", "RAG", "SelfCheckGPT", "VeriLLM", "ReLiC", "VICTOR (Ours)"]
    
    # Reorder
    overall = overall.set_index('Framework').loc[desired_framework_order].reset_index()
    
    print("\n\n" + "=" * 70)
    print("OVERALL AVERAGES")
    print("=" * 70)
    print("Framework\tPrecision\tRecall\tF1 Score\tHall Rate")
    for _, row in overall.iterrows():
        print(f"{row['Framework']}\t{row['Precision']}\t{row['Recall']}\t{row['F1 Score']}\t{row['Hallucination Rate']}")
    
    # Ablation
 
    # ============================================================
    fever_split = len(fever) // 2
    fever_train, fever_test = fever[:fever_split], fever[fever_split:]
    
    # Build from TRAINING claims only
    ref_docs = [c for c, l, _ in fever_train if l == 0]
    ds_rc, ds_rt = build_unigram(ref_docs) if ref_docs else ({}, 1)
    fever_bm25, fever_bm25_corpus = build_bm25_index(ref_docs) if ref_docs else (None, None)
    
    print(f"\n  Ablation on FEVER: Train={len(fever_train)}, Test={len(fever_test)}, BM25 docs={len(ref_docs)}")
    
    full_res = evaluate_framework("VICTOR (Full)", victor_full, fever_test, ref_docs, ds_rc, ds_rt, True, fever_bm25, fever_bm25_corpus)
    no_unc_res = evaluate_framework("VICTOR (No Uncertainty)", victor_no_unc, fever_test, ref_docs, ds_rc, ds_rt, False, fever_bm25, fever_bm25_corpus)
    delta_f1 = ((full_res["F1 Score"] - no_unc_res["F1 Score"]) / max(0.001, no_unc_res["F1 Score"])) * 100
    
    print("\n\n" + "=" * 70)
    print("ABLATION: VICTOR With vs Without Uncertainty (FEVER)")
    print("=" * 70)
    print("Variant\tPrecision\tRecall\tF1 Score\tHall Rate\t95% CI\tΔ (%)")
    print(f"VICTOR (No Uncertainty)\t{no_unc_res['Precision']}\t{no_unc_res['Recall']}\t{no_unc_res['F1 Score']}\t{no_unc_res['Hallucination Rate']}\t{no_unc_res['F1 CI']}\t—")
    print(f"VICTOR (Full)\t{full_res['Precision']}\t{full_res['Recall']}\t{full_res['F1 Score']}\t{full_res['Hallucination Rate']}\t{full_res['F1 CI']}\t{delta_f1:.1f}")
    
    # ============================================================
    # SAVE TO EXCEL (with all 6 frameworks)
    # ============================================================
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"results/VICTOR_{LLM_MODEL.replace(':', '_')}_S{SAMPLE_PER_DATASET}_T{UNCERTAINTY_THRESHOLD}_{timestamp}.xlsx"
    
    # Debug: Check frameworks
    print("\n" + "=" * 70)
    print("DEBUG: Frameworks in DataFrame before saving")
    print("=" * 70)
    print(df['Framework'].unique().tolist())
    
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        startrow = 0
        
        # Config table
        config_data = pd.DataFrame([
            {"Parameter": "SAMPLE_PER_DATASET", "Value": SAMPLE_PER_DATASET, "Note": "Total records loaded per source"},
            {"Parameter": "TAU (Uncertainty Gate)", "Value": UNCERTAINTY_THRESHOLD, "Note": "Threshold for LLM trigger"},
            {"Parameter": "SEED", "Value": SEED, "Note": "For reproducibility"},
            {"Parameter": "BM25", "Value": "Enabled", "Note": "Real retriever"},
            {"Parameter": "Total Test Claims", "Value": total_test_claims, "Note": "50% split used for testing"}
        ])
        config_data.to_excel(writer, sheet_name="All_Results", startrow=startrow, index=False)
        startrow += len(config_data) + 3
        
        # Per-dataset table - ALL frameworks
        column_order = ["Dataset", "Framework", "Precision", "Recall", "F1 Score", "Hallucination Rate", "F1 CI"]
        df_sorted = df[column_order].sort_values(["Dataset", "Framework"])
        df_sorted.to_excel(writer, sheet_name="All_Results", startrow=startrow, index=False)
        startrow += len(df_sorted) + 3
        

        # Pivot table
        pivot = df.pivot(index="Dataset", columns="Framework", values="F1 Score").reset_index()
        print("DEBUG Excel: Pivot columns before reorder:", pivot.columns.tolist())

        # Force correct column order
        desired_cols = ["Dataset", "FActScore", "RAG", "SelfCheckGPT", "VeriLLM", "ReLiC", "VICTOR (Ours)"]
        existing_cols = [col for col in desired_cols if col in pivot.columns]
        pivot = pivot[existing_cols]
        
        print("DEBUG Excel: Pivot columns after reorder:", pivot.columns.tolist())
        
        pivot.to_excel(writer, sheet_name="All_Results", startrow=startrow, index=False)
        startrow += len(pivot) + 3
        
        # Overall averages
        # Overall averages (with correct order)
        overall = df.groupby("Framework").agg({"Precision": "mean", "Recall": "mean", "F1 Score": "mean", "Hallucination Rate": "mean"}).round(3).reset_index()
        
        # Define desired framework order
        fw_order = ["FActScore", "RAG", "SelfCheckGPT", "VeriLLM", "ReLiC", "VICTOR (Ours)"]
        
        # Reorder
        overall = overall.set_index('Framework').loc[fw_order].reset_index()
        
        overall.to_excel(writer, sheet_name="All_Results", startrow=startrow, index=False)
        startrow += len(overall) + 3
        
        # Ablation table
        ablation_df = pd.DataFrame([
            {"Variant": "VICTOR (No Uncertainty)", "Precision": no_unc_res["Precision"], 
             "Recall": no_unc_res["Recall"], "F1 Score": no_unc_res["F1 Score"], 
             "Hall Rate": no_unc_res["Hallucination Rate"], "F1 CI": no_unc_res["F1 CI"], "Δ (%)": "—"},
            {"Variant": "VICTOR (Full)", "Precision": full_res["Precision"], 
             "Recall": full_res["Recall"], "F1 Score": full_res["F1 Score"], 
             "Hall Rate": full_res["Hallucination Rate"], "F1 CI": full_res["F1 CI"], "Δ (%)": f"{delta_f1:.1f}"}
        ])
        ablation_df.to_excel(writer, sheet_name="All_Results", startrow=startrow, index=False)
    
    print("\n" + "=" * 70)
    print(f"✅ Saved: {output_path}")
    print(f"Frameworks saved: {df['Framework'].unique().tolist()}")
    print("=" * 70)

if __name__ == "__main__":
    main()

C:\Users\77sam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



VICTOR - Verification with Intelligent Confidence Thresholding and Optimal Retrieval
Model: gemma3:1b | Samples: 500 | τ: 0.3
BM25: Enabled (Training-only) | SelfCheckGPT: Available

Parameter	Value	Note
-------------------------------------------------------
SAMPLE_PER_DATASET	500	Total records loaded per source
TAU (Uncertainty Gate)	0.3	Threshold for LLM trigger
SEED	42	For reproducibility
Total Test Claims	750	50% split used for testing

  Processing FEVER...
    Training claims: 250 | Test claims: 250 | BM25 docs: 192
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initialized
SelfCheck-1gram initia

In [2]:
# ============================================================
# NEW ABLATION — NO_BM25, NO_REVISION, NO_ABSTENTION
# Run after main() completes
# ============================================================

def victor_no_bm25(claim, docs, rc, rt, use_uncertainty, bm25_index, bm25_corpus):
    """VICTOR without BM25 — uses training docs directly, no retrieval"""
    parts = re.split(r'[.!?]+', claim)
    parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
    preds = []
    for p in parts:
        if not p:
            continue
        if use_uncertainty:
            uncertainty = get_predictive_entropy(p)
            if uncertainty > UNCERTAINTY_THRESHOLD:
                pred = call_llm_verify(p, docs)  # no BM25, use training docs directly
            else:
                pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        else:
            pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        preds.append(pred)
    return 1 if any(preds) else 0

def victor_no_revision(claim, docs, rc, rt, use_uncertainty, bm25_index, bm25_corpus):
    """VICTOR without revision — REFUTED stays REFUTED"""
    parts = re.split(r'[.!?]+', claim)
    parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
    preds = []
    for p in parts:
        if not p:
            continue
        if use_uncertainty:
            uncertainty = get_predictive_entropy(p)
            if uncertainty > UNCERTAINTY_THRESHOLD:
                if bm25_index:
                    retrieved = retrieve_bm25(p, bm25_index, bm25_corpus, top_k=3)
                    pred = call_llm_verify(p, retrieved)
                else:
                    pred = call_llm_verify(p, docs)
            else:
                pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        else:
            pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        preds.append(pred)
    return 1 if any(preds) else 0

def victor_no_abstention(claim, docs, rc, rt, use_uncertainty, bm25_index, bm25_corpus):
    """VICTOR without abstention — always attempts revision even with low overlap"""
    parts = re.split(r'[.!?]+', claim)
    parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
    preds = []
    for p in parts:
        if not p:
            continue
        if use_uncertainty:
            uncertainty = get_predictive_entropy(p)
            if uncertainty > UNCERTAINTY_THRESHOLD:
                if bm25_index:
                    retrieved = retrieve_bm25(p, bm25_index, bm25_corpus, top_k=3)
                    pred = call_llm_verify(p, retrieved)
                else:
                    pred = call_llm_verify(p, docs)
            else:
                pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        else:
            pred = 0 if evidence_overlap(p, docs) >= OVERLAP_THRESHOLD else 1
        preds.append(pred)
    return 1 if any(preds) else 0

# Load datasets (reuse same seed/split as main)
fever2  = load_fever(FEVER_PATH)
halu2   = load_halueval(HALU_PATH)
truth2  = load_truthfulqa(TRUTH_PATH)
datasets2 = {"FEVER": fever2, "HaluEval": halu2, "TruthfulQA": truth2}

new_frameworks = {
    "VICTOR (No BM25)":       victor_no_bm25,
    "VICTOR (No Revision)":   victor_no_revision,
    "VICTOR (No Abstention)": victor_no_abstention,
}

new_results = []
for ds_name, ds_data in datasets2.items():
    print(f"\n Processing {ds_name}...")
    split = len(ds_data) // 2
    train, test = ds_data[:split], ds_data[split:]
    train_ref = [c for c, l, _ in train if l == 0]
    ds_rc, ds_rt = build_unigram(train_ref) if train_ref else ({}, 1)
    bm25_idx, bm25_corp = build_bm25_index(train_ref) if train_ref else (None, None)

    for fw_name, fw_fn in new_frameworks.items():
        res = evaluate_framework(
            fw_name, fw_fn, test, train_ref,
            ds_rc, ds_rt, True, bm25_idx, bm25_corp
        )
        res["Dataset"] = ds_name
        new_results.append(res)
        print(f"   {fw_name}: P={res['Precision']} R={res['Recall']} F1={res['F1 Score']}")

new_df = pd.DataFrame(new_results)
new_df.to_csv("results/ablation_new_variants.csv", index=False)
print("\nSaved: results/ablation_new_variants.csv")
print(new_df[["Dataset","Framework","Precision","Recall","F1 Score"]].to_string())


 Processing FEVER...
   VICTOR (No BM25): P=0.217 R=0.981 F1=0.356
   VICTOR (No Revision): P=0.231 R=0.963 F1=0.373
   VICTOR (No Abstention): P=0.231 R=0.963 F1=0.373

 Processing HaluEval...
   VICTOR (No BM25): P=0.5 R=0.942 F1=0.653
   VICTOR (No Revision): P=0.575 R=0.892 F1=0.699
   VICTOR (No Abstention): P=0.575 R=0.892 F1=0.699

 Processing TruthfulQA...
   VICTOR (No BM25): P=0.807 R=0.893 F1=0.848
   VICTOR (No Revision): P=0.821 R=0.934 F1=0.874
   VICTOR (No Abstention): P=0.821 R=0.934 F1=0.874

Saved: results/ablation_new_variants.csv
      Dataset               Framework  Precision  Recall  F1 Score
0       FEVER        VICTOR (No BM25)      0.217   0.981     0.356
1       FEVER    VICTOR (No Revision)      0.231   0.963     0.373
2       FEVER  VICTOR (No Abstention)      0.231   0.963     0.373
3    HaluEval        VICTOR (No BM25)      0.500   0.942     0.653
4    HaluEval    VICTOR (No Revision)      0.575   0.892     0.699
5    HaluEval  VICTOR (No Abstention)   

In [3]:
# ============================================================
# TAU AND TOP-K SENSITIVITY SWEEP — FEVER ONLY
# Run after ablation cell completes
# Same seed, same 500 samples, same model
# Only τ or k changes — everything else fixed
# ============================================================

import json, os

SWEEP_RESULTS_PATH = "results/sensitivity_results.json"

# Load FEVER with same seed/sample as original
fever_sweep = load_fever(FEVER_PATH)
split = len(fever_sweep) // 2
fever_train, fever_test = fever_sweep[:split], fever_sweep[split:]
train_ref = [c for c, l, _ in fever_train if l == 0]
ds_rc, ds_rt = build_unigram(train_ref) if train_ref else ({}, 1)

print(f"FEVER sweep: Train={len(fever_train)}, Test={len(fever_test)}, BM25 docs={len(train_ref)}")

sweep_results = {}

# ── τ Sweep ───────────────────────────────────────────────────
print("\n" + "="*60)
print("TAU SWEEP (k=3 fixed)")
print("="*60)

tau_results = []
for tau_val in [0.1, 0.2, 0.3, 0.4, 0.5]:
    # Build fresh BM25 index (same every time)
    bm25_idx, bm25_corp = build_bm25_index(train_ref)

    y_true, y_pred = [], []
    for claim, label, docs in fever_test:
        parts = re.split(r'[.!?]+', claim)
        parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
        preds = []
        for p in parts:
            if not p:
                continue
            uncertainty = get_predictive_entropy(p)
            if uncertainty > tau_val:
                retrieved = retrieve_bm25(p, bm25_idx, bm25_corp, top_k=3)
                pred = call_llm_verify(p, retrieved)
            else:
                pred = 0 if evidence_overlap(p, train_ref) >= OVERLAP_THRESHOLD else 1
            preds.append(pred)
        y_true.append(label)
        y_pred.append(1 if any(preds) else 0)

    prec, rec, f1 = safe_metrics(y_true, y_pred)
    ci_lo, ci_hi = bootstrap_ci(y_true, y_pred, n_bootstrap=500)
    row = {
        "tau": tau_val, "precision": round(prec,3),
        "recall": round(rec,3), "f1": round(f1,3),
        "ci": f"[{ci_lo:.3f}-{ci_hi:.3f}]"
    }
    tau_results.append(row)
    print(f"  τ={tau_val}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} CI={row['ci']}")

sweep_results["tau_sweep"] = tau_results

# ── k Sweep ───────────────────────────────────────────────────
print("\n" + "="*60)
print("TOP-K SWEEP (τ=0.3 fixed)")
print("="*60)

topk_results = []
for k_val in [1, 3, 5]:
    bm25_idx, bm25_corp = build_bm25_index(train_ref)

    y_true, y_pred = [], []
    for claim, label, docs in fever_test:
        parts = re.split(r'[.!?]+', claim)
        parts = [p.strip() for p in parts if len(p.strip()) > 15] or [claim]
        preds = []
        for p in parts:
            if not p:
                continue
            uncertainty = get_predictive_entropy(p)
            if uncertainty > UNCERTAINTY_THRESHOLD:
                retrieved = retrieve_bm25(p, bm25_idx, bm25_corp, top_k=k_val)
                pred = call_llm_verify(p, retrieved)
            else:
                pred = 0 if evidence_overlap(p, train_ref) >= OVERLAP_THRESHOLD else 1
            preds.append(pred)
        y_true.append(label)
        y_pred.append(1 if any(preds) else 0)

    prec, rec, f1 = safe_metrics(y_true, y_pred)
    ci_lo, ci_hi = bootstrap_ci(y_true, y_pred, n_bootstrap=500)
    row = {
        "top_k": k_val, "precision": round(prec,3),
        "recall": round(rec,3), "f1": round(f1,3),
        "ci": f"[{ci_lo:.3f}-{ci_hi:.3f}]"
    }
    topk_results.append(row)
    print(f"  k={k_val}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} CI={row['ci']}")

sweep_results["topk_sweep"] = topk_results

# ── Save ──────────────────────────────────────────────────────
with open(SWEEP_RESULTS_PATH, "w") as f:
    json.dump(sweep_results, f, indent=2)

# Also save as CSV for easy use
tau_df  = pd.DataFrame(tau_results)
topk_df = pd.DataFrame(topk_results)
tau_df.to_csv("results/tau_sensitivity.csv", index=False)
topk_df.to_csv("results/topk_sensitivity.csv", index=False)

print("\n" + "="*60)
print("Saved:")
print("  results/sensitivity_results.json")
print("  results/tau_sensitivity.csv")
print("  results/topk_sensitivity.csv")
print("="*60)
print("\nτ=0.3 should show best or near-best F1 to justify paper choice.")
print("k=3 should show best or near-best F1 to justify paper choice.")

FEVER sweep: Train=250, Test=250, BM25 docs=178

TAU SWEEP (k=3 fixed)
  τ=0.1: P=0.287 R=0.985 F1=0.444 CI=[0.374-0.511]
  τ=0.2: P=0.287 R=0.985 F1=0.444 CI=[0.372-0.511]
  τ=0.3: P=0.287 R=0.985 F1=0.444 CI=[0.373-0.515]
  τ=0.4: P=0.287 R=0.985 F1=0.444 CI=[0.370-0.515]
  τ=0.5: P=0.296 R=0.881 F1=0.444 CI=[0.369-0.515]

TOP-K SWEEP (τ=0.3 fixed)
  k=1: P=0.290 R=0.955 F1=0.444 CI=[0.368-0.516]
  k=3: P=0.287 R=0.985 F1=0.444 CI=[0.371-0.511]
  k=5: P=0.287 R=0.985 F1=0.444 CI=[0.375-0.506]

Saved:
  results/sensitivity_results.json
  results/tau_sensitivity.csv
  results/topk_sensitivity.csv

τ=0.3 should show best or near-best F1 to justify paper choice.
k=3 should show best or near-best F1 to justify paper choice.
